In [1]:
!pip install torchdiffeq
!pip install pysr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pysr]5/6 [pysr]


In [5]:
import time
import numpy as np
import torch

import os, math
import torch.nn as nn
from torchdiffeq import odeint
import sys

from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from project.models.excitation import UFunFromSamples
from project.models.pinode_linear_3dof import PINODEFuncLinear3DOF
from project.models.pinode_nsd_3dof import PINODEFuncNSD_3DOF
from project.models.truth_nsd_3dof import TruthPhaseNSD_3DOF
from project.models.nsd_net import NSD_Net


device = "cpu"

# Parameters
c1 = c2 = c3 = 0.015
m1 = m2 = 9.0
m3 = 9.2


def model_accel(model, t, x, v):
    h = torch.cat([x, v], dim=-1)
    dh = model(t, h)
    ndof = x.shape[-1]
    return dh[..., ndof:]


def rollout_central_difference(model, h0, t_grid):
    dt = (t_grid[1] - t_grid[0]).to(h0)
    T = t_grid.numel()

    if h0.ndim == 1:
        h0 = h0.unsqueeze(0)
        batch = False
    else:
        batch = True

    B, D = h0.shape
    ndof = D // 2

    x0 = h0[:, :ndof]
    v0 = h0[:, ndof:]

    x_prev = x0 - dt * v0

    traj = torch.zeros((T, B, D), device=h0.device, dtype=h0.dtype)
    traj[0, :, :ndof] = x0
    traj[0, :, ndof:] = v0

    x = x0
    for n in range(T - 1):
        t = t_grid[n]
        v = (x - x_prev) / dt
        a = model_accel(model, t, x, v)

        x_next = 2 * x - x_prev + (dt ** 2) * a
        v_next = (x_next - x) / dt

        traj[n + 1, :, :ndof] = x_next
        traj[n + 1, :, ndof:] = v_next

        x_prev, x = x, x_next

    return traj if batch else traj[:, 0, :]


# Stiffness matrix K for a 3-DOF
K = torch.tensor([
    [20.26,   -11.16,   0.0],
    [-11.16,  20.33,    -9.17],
    [0.0,      -9.17,     9.17],
], dtype=torch.float32, device=device)

# Damping matrix (diagonal with c1..c3)
C = torch.diag(torch.tensor([c1, c2, c3], dtype=torch.float32, device=device))

# Mass Matrix
M = (1.0/386) * torch.diag(torch.tensor([m1, m2, m3], dtype=torch.float32, device=device))

r = torch.ones(3, device=device)
B = -(M @ r)  # (3,)

# Start from nominal
M_true = M.clone()
K_true = K.clone()
C_true = C.clone()

# Add controlled mismatch
K_true = 0.97 * K_true  # stiffness is 3% lower than nominal
C_true = 1.25 * C_true  # damping is 25% higher than nominal

# Initial conditions (NOTE: use 6-dim states for 3DOF)
h0_4 = torch.tensor([0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=torch.float32, device=device)


def nsd_force_bilinear(x1, d0=0.15, k_post=-10.0):
    excess = torch.relu(torch.abs(x1) - d0)
    return k_post * excess * torch.sign(x1)


# ============================================================
# El Centro excitation: load + resample to dt=1/256
# ============================================================
data = np.loadtxt("../data/elcentro.dat")
dt_ec = 0.02

if data.ndim == 1:
    acc_ec = data.astype(np.float32)
    t_ec = (np.arange(len(acc_ec), dtype=np.float32) * dt_ec)
else:
    t_ec = data[:, 0].astype(np.float32)
    acc_ec = data[:, 1].astype(np.float32)

T_end = float(t_ec[-1])

dt = 1.0 / 256.0
t_full = torch.arange(0.0, T_end + dt, dt, device=device)

u_full_np = np.interp(
    t_full.detach().cpu().numpy(),
    t_ec,
    acc_ec
).astype(np.float32)
u_full = torch.tensor(u_full_np, device=device)

T_train_sec = 15.0
N_train = int(T_train_sec / dt) + 1
t_train = t_full[:N_train]
u_train = u_full[:N_train]
# at2_path = "/content/drive/MyDrive/20251212_1765548540.60025_1/ath.KOBE.KBU000.AT2"

# def read_at2(filepath):
#     with open(filepath, "r") as f:
#         lines = f.readlines()

#     parts = lines[3].split()
#     npts = int(parts[0])
#     dt_file = float(parts[1])

#     data_str = " ".join(lines[4:])
#     acc = np.fromstring(data_str, sep=" ")
#     acc = acc[:npts]
#     return acc.astype(np.float32), dt_file, npts

# u_file_np, dt_file, npts = read_at2(at2_path)

# # File's physical timeline (dt_file = 0.01, NPTS=3200 -> ~31.99s)
# T_end = (npts - 1) * dt_file
# t_file = torch.arange(0.0, T_end + dt_file, dt_file)   # length npts

# # Paper/integration grid
# dt = 1.0 / 256.0
# t_full = torch.arange(0.0, T_end + dt, dt, device=device)  # 8193 points

# # Resample acceleration onto t_full (linear interpolation)
# u_full_np = np.interp(t_full.detach().cpu().numpy(),
#                       t_file.numpy(),
#                       u_file_np).astype(np.float32)

# u_full = torch.tensor(u_full_np, device=device)


# T_train_sec = 15.0
# N_train = int(T_train_sec / dt) + 1
# t_train = t_full[:N_train]
# u_train = u_full[:N_train]


# Load models and run truth trajectory

model = PINODEFuncLinear3DOF(M, K, C, B)
true_trajectory = TruthPhaseNSD_3DOF(M_true, K_true, C_true, nsd_force_bilinear)

true_trajectory.u_fun = UFunFromSamples(t_train, u_train)
true_trajectory.amp =120.0


save_path = "../models/"

save_model_path_nn1 = save_path + "linear_with_forced_vibration_base_model_15sec.pth"
model.load_state_dict(torch.load(save_model_path_nn1, map_location=device))
model.amp = 120.0
model.u_fun = UFunFromSamples(t_train, u_train)

nn2 = NSD_Net().to(device)
model_nsd = PINODEFuncNSD_3DOF(M, K, C, model.mlp, nn2).to(device)


save_model_path_nn2 = save_path + "nsd_forced_vibration_base_15sec_model.pth"
model_nsd.load_state_dict(torch.load(save_model_path_nn2, map_location=device))
for p in model_nsd.mlp.parameters():
    p.requires_grad = False
model_nsd.u_fun = UFunFromSamples(t_train, u_train)
model_nsd.amp = 120.0


with torch.no_grad():
    traj = odeint(true_trajectory, h0_4, t_train, method="rk4")  # (T,6)


# Build dataset: h = truth states, y = NN2 force on DOF1
if isinstance(traj, (list, tuple)):
    trajs = list(traj)
else:
    trajs = [traj]

y_targets = []
x1_all = []

with torch.no_grad():
    for h in trajs:
        h = h.to(device)
        if h.ndim != 2 or h.shape[1] != 6:
            raise ValueError(f"Expected (T,6), got {tuple(h.shape)}")

        x1 = h[:, 0]

        a_raw = model_nsd.nsd(h).squeeze(-1)  # (T,)
        a_vec = torch.zeros((h.shape[0], 3), device=device, dtype=h.dtype)
        a_vec[:, 0] = a_raw

        f_vec = (M @ a_vec.unsqueeze(-1)).squeeze(-1)  # (T,3)
        f1 = f_vec[:, 0]  # (T,)

        # x1_all.append(x1.detach().cpu().numpy())
        y_targets.append(f1.detach().cpu().numpy())


x1 = traj[:,0].cpu().numpy()
d0 = 0.15
print("frac(|x1|>d0) =", np.mean(np.abs(x1)>d0))

/tmp/ipykernel_20651/592799612.py:193: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_model_path_nn1, map_location=device))
/tmp/ipykern

frac(|x1|>d0) = 0.3355896901848477


In [8]:
import numpy as np
import torch
from pysr import PySRRegressor


# Build dataset
H_all = traj.detach().cpu().numpy() if torch.is_tensor(traj) else np.asarray(traj)

if isinstance(y_targets, (list, tuple)):
    y_all = np.concatenate(y_targets, axis=0)
else:
    y_all = y_targets.detach().cpu().numpy() if torch.is_tensor(y_targets) else np.asarray(y_targets)

y_all = y_all.reshape(-1)

assert H_all.ndim == 2 and H_all.shape[1] == 6, f"traj must be (N,6), got {H_all.shape}"
assert y_all.ndim == 1 and y_all.shape[0] == H_all.shape[0], f"y must be (N,), got {y_all.shape}"

print("Dataset:", H_all.shape, y_all.shape)
print("y stats: mean=%.6g std=%.6g min=%.6g max=%.6g" % (
    float(y_all.mean()), float(y_all.std()), float(y_all.min()), float(y_all.max())
))

# ALL_NAMES = ["x1", "x2", "x3", "v1", "v2", "v3"]

# # ===========
# # 2) Features
# # ===========
# feat_idx = [0,1,2,3,4,5]
# feature_names = [ALL_NAMES[j] for j in feat_idx]

# X = H_all[:, feat_idx].astype(np.float64)


x1, x2, x3 = H_all[:, 0], H_all[:, 1], H_all[:, 2]
v1, v2, v3 = H_all[:, 3], H_all[:, 4], H_all[:, 5]

# drifts
d1 = x1
d2 = x2 - x1
d3 = x3 - x2

# drift velocities
w1 = v1
w2 = v2 - v1
w3 = v3 - v2

ALL_NAMES = ["d1", "d2", "d3", "w1", "w2", "w3"]

# Features
feat_idx = [0, 1, 2, 3, 4, 5]  # all drifts + drift-vels
feature_names = [ALL_NAMES[j] for j in feat_idx]

H_feat = np.column_stack([d1, d2, d3, w1, w2, w3]).astype(np.float64)
X = H_feat[:, feat_idx]
y = y_all.astype(np.float64)

# Train on all data
X_tr, y_tr = X, y

# PySR: loss-first settings
BASE_KW = dict(
    niterations=200,
    populations=50,
    population_size=300,
    maxsize=10,
    maxdepth=7,
    parsimony=0.0,
    binary_operators=["+", "-", "*"],
    unary_operators=["abs", "sign", "relu"],
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    model_selection="best",
    parallelism="multithreading",
    deterministic=False,
    verbosity=1,
    random_state=42
)

model = PySRRegressor(**BASE_KW)
model.fit(X_tr, y_tr)

hof = model.get_hof()
row = hof.loc[hof["loss"].idxmin()]

loss = float(row["loss"])
eq = row["equation"]

# Map x0,x1,... to names
eq_named = eq
for i in reversed(range(len(feature_names))):
    eq_named = eq_named.replace(f"x{i}", feature_names[i])

print("\nEquation (named):")
print(eq_named)

# Evaluate training MSE/RMSE with the best model
yhat = model.predict(X)
mse_raw = float(np.mean((yhat - y) ** 2))
rmse_raw = float(np.sqrt(mse_raw))
print("\nTraining fit:")
print("MSE:", mse_raw)
print("RMSE:", rmse_raw)

# Save
# with open("best_pysr_equation_on_my_data.txt", "w") as f:
#     f.write(eq_named + "\n")

# print("\nSaved best_pysr_equation_on_my_data.txt")

Dataset: (3841, 6) (3841,)
y stats: mean=-0.0102995 std=0.522794 min=-2.23522 max=2.26538


/home/sdi2100218/Thesis/Interpretable-Structural-Dynamics/.venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/home/sdi2100218/Thesis/Interpretable-Structural-Dynamics/.venv/lib/python3.11/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.040e+05
Progress: 119 / 10000 total iterations (1.190%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.733e-01  0.000e+00  y = -0.010302
3           8.128e-02  6.064e-01  y = x₀ * -2.957
5           7.810e-02  1.996e-02  y = (-2.1513 * x₀) - x₁
8           7.532e-02  1.207e-02  y = (x₀ * (-2.7317 - abs(x₀))) + -0.0053506
10          6.799e-02  5.120e-02  y = (-0.93851 - relu(relu(abs(x₀)))) * (x₀ * 2.4058)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 2.120e+05
Progress: 250 / 10000 total iterations (2.50

[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.070e+05
Progress: 9770 / 10000 total iterations (97.700%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.733e-01  0.000e+00  y = -0.010302
3           8.128e-02  6.064e-01  y = x₀ * -2.957
5           7.808e-02  2.006e-02  y = (x₀ * -2.1778) - x₁
6           2.224e-02  1.256e+00  y = abs(x₀) * (x₀ * -14.222)
7           1.264e-02  5.650e-01  y = ((x₀ * x₀) * -52.322) * x₀
8           9.294e-03  3.074e-01  y = x₀ * ((abs(x₀) - 0.10081) * -23.77)
9           7.824e-03  1.722e-01  y = relu(abs(x₀) - 0.11495) * (x₀ * -26.047)
10          5.725e-03  3.124e-01  y = (relu(abs(x₀) - 0.15207) * sign(x₀)) * -9.8466
───────────────────────────────────────────────────────────────────────────────────────────────────
═══════════════════════════════════════